# Indian TTS — Staged Training with Podcast Validation

Train Indian English TTS (male & female) step by step, with a **fixed podcast script** generated at every stage so you can hear the improvement.

| Stage | Time | Cumulative | Podcast Sounds Like |
|-------|------|-----------|--------------------|
| **0** | ~2 min | 2 min | Static/noise |
| **1** | ~30 min | 32 min | Buzzy, some energy |
| **2** | ~1 hr | 1 hr | Speech-like noise |
| **3** | ~1 hr | 2 hrs | Vowel sounds, patterns |
| **4** | ~6 hrs | 8 hrs | Partially intelligible words |
| **5** | ~20 hrs | 28 hrs | Clear Indian English |

**The same AI podcast (Arjun & Priya discussing AI in India) is generated at every stage.** Compare them side by side.

## Compute Budget

| What | Cost |
|------|------|
| Colab Pro+ | $49.99/month (flat) |
| Stages 0-3 (validation) | ~2 hrs A100 |
| Stage 4 (good quality) | ~6 more hrs |
| Stage 5 (best quality) | ~20 more hrs |
| **Total for Stage 4** | **~8 hrs, fits in 1 session** |

**Requirements:** Colab Pro+ with A100 GPU

---
## Setup (run once)

In [18]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
NVIDIA A100-SXM4-80GB, 81920 MiB
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85 GB


In [19]:
%cd /content
!rm -rf /content/indian_tts

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [20]:
# Clone repo and install
!git clone https://github.com/seetha0712/text2speech_1.git /content/indian_tts 2>/dev/null || echo "Already cloned"
%cd /content/indian_tts
!git checkout claude/custom-indian-tts-model-TUAjJ
!git pull origin claude/custom-indian-tts-model-TUAjJ
!pip install -q -r requirements.txt
!pip install -q -e .
!apt-get install -qq espeak-ng > /dev/null 2>&1
print("Setup complete!")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Already cloned
[Errno 2] No such file or directory: '/content/indian_tts'
/content/indian_tts
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or director

In [ ]:
# Download Indian English data (all CC-0 / CC-BY 4.0 — legally safe)
# Svarah is guaranteed to work (~9.6 hrs Indian English)
# Common Voice mirror will be tried as bonus (larger dataset)
!python -m indian_tts.data.preprocess \
    --source all \
    --output /content/data \
    --max-hours 15 \
    --min-upvotes 2

In [ ]:
# Download Indian English data (CC-0 + CC-BY 4.0)
!python -m indian_tts.data.preprocess \
    --source all \
    --output /content/data \
    --max-hours 15 \
    --min-upvotes 2

In [ ]:
import yaml, os

with open('configs/colab_a100_config.yaml') as f:
    config = yaml.safe_load(f)

config['data']['training_files'] = '/content/data/train.txt'
config['data']['validation_files'] = '/content/data/val.txt'
config['paths']['output_dir'] = '/content/outputs'
config['paths']['checkpoint_dir'] = '/content/outputs/checkpoints'
config['paths']['log_dir'] = '/content/outputs/logs'

with open('/content/stage_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config ready!")
for f in ['train.txt', 'val.txt']:
    path = f'/content/data/{f}'
    if os.path.exists(path):
        with open(path) as fh:
            lines = [l for l in fh if l.strip() and not l.startswith('#')]
        print(f"{f}: {len(lines)} samples")

### Helper: Listen to the podcast from any stage

In [ ]:
import IPython.display as ipd
import glob

def listen_podcast(stage_num):
    """Play the full podcast and individual lines from a given stage."""
    podcast_dir = f'/content/outputs/podcast_stage_{stage_num}'
    full_path = f'{podcast_dir}/podcast_full.wav'

    if not os.path.exists(full_path):
        print(f"No podcast found for stage {stage_num}.")
        return

    print(f"{'=' * 60}")
    print(f"  PODCAST — Stage {stage_num}")
    print(f"  Arjun (male) & Priya (female) discuss AI in India")
    print(f"{'=' * 60}")
    print(f"\nFull podcast:")
    ipd.display(ipd.Audio(full_path))

    # Also show first 4 individual lines so you can hear each speaker
    lines = sorted(glob.glob(f'{podcast_dir}/line_*.wav'))[:4]
    script_lines = [
        ('Priya', 'Welcome to AI India, the podcast where we explore...'),
        ('Arjun', 'And I am Arjun. Today we are talking about...'),
        ('Priya', 'India now has over three hundred AI startups...'),
        ('Arjun', 'Many of these companies are solving uniquely Indian problems...'),
    ]
    if lines:
        print(f"\nFirst few lines (to hear each speaker):")
        for wav_path, (speaker, preview) in zip(lines, script_lines):
            print(f"\n  [{speaker}] {preview}")
            ipd.display(ipd.Audio(wav_path))

def compare_podcasts():
    """Play all available stage podcasts side by side."""
    print("COMPARE: How the podcast improves across stages\n")
    for stage in range(6):
        full_path = f'/content/outputs/podcast_stage_{stage}/podcast_full.wav'
        if os.path.exists(full_path):
            import soundfile as sf
            data, sr = sf.read(full_path)
            duration = len(data) / sr
            print(f"\nStage {stage} ({duration:.0f}s):")
            ipd.display(ipd.Audio(full_path))

print("Helpers loaded: listen_podcast(stage_num), compare_podcasts()")

---
## Stage 0 — Sanity Check (~2 min)

Does it run without crashing? Podcast will be **pure noise** — that's expected.

In [ ]:
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 0

In [ ]:
listen_podcast(0)  # Expect: noise/static

**PASSED?** Proceed to Stage 1. **FAILED?** Check errors above (OOM → reduce batch_size to 32).

---
## Stage 1 — Smoke Test (~30 min)

Are losses decreasing? Podcast will be **buzzy/noisy but not silent**.

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
print(f"Resuming from: {resume}")
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 1 --resume {resume}

In [ ]:
listen_podcast(1)  # Expect: buzzy, some energy, not pure static

**PASSED?** Proceed to Stage 2. **FAILED?** Lower learning_rate to 0.0001.

---
## Stage 2 — Early Signal (~1 hr cumulative)

Is audio structure emerging? Podcast will have **speech-like noise**.

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 2 --resume {resume}

In [ ]:
listen_podcast(2)  # Expect: speech-like noise, garbled
print("\n--- Compare stages 0 vs 1 vs 2: ---")
compare_podcasts()

**PASSED + male/female differ?** Proceed to Stage 3. **FAILED?** Increase --max-hours.

---
## Stage 3 — Quality Gate (~2 hrs cumulative) — THE GO/NO-GO POINT

Do different texts produce different outputs? Are Arjun and Priya distinguishable?

**If this passes, the 6-hour investment for Stage 4 is worth it.**

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 3 --resume {resume}

In [ ]:
listen_podcast(3)  # Expect: vowel sounds, speech patterns
print("\n--- Compare all stages so far: ---")
compare_podcasts()

### DECISION POINT

**PASSED?** The model IS learning. Stage 4 (6 hrs) will produce intelligible speech. **Go for it.**

**FAILED?** STOP. Do not spend 6 more hours. Try:
- Re-download data with `--min-upvotes 3` and `--max-hours 25`
- Reduce batch_size to 32 and restart from scratch

---
## Stage 4 — Full Training (~8 hrs cumulative)

First intelligible speech. **Back up to Google Drive first!**

In [ ]:
# Back up checkpoints to Google Drive
from google.colab import drive
import shutil
drive.mount('/content/drive')
drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
os.makedirs(drive_backup, exist_ok=True)
for ckpt in sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))[-2:]:
    shutil.copy2(ckpt, drive_backup)
    print(f"Backed up: {os.path.basename(ckpt)}")
shutil.copy2('/content/stage_config.yaml', drive_backup)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/outputs/logs

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
print(f"Resuming from: {resume}")
print("This will take ~6 hours...")
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 4 --resume {resume}

In [ ]:
listen_podcast(4)  # Expect: partially intelligible speech!
print("\n--- Full comparison across all stages: ---")
compare_podcasts()

# Backup to Drive
for ckpt in sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))[-2:]:
    shutil.copy2(ckpt, drive_backup)
# Also backup the podcast samples
for stage in range(5):
    src = f'/content/outputs/podcast_stage_{stage}/podcast_full.wav'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_backup, f'podcast_stage_{stage}.wav'))
print("All backed up to Google Drive!")

---
## Stage 5 — Extended Training (~28 hrs cumulative, optional)

For highest quality. May need multiple Colab sessions — resume from Drive.

In [ ]:
# Restore from Drive if this is a new session
import glob, os, shutil
drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
local_ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
drive_ckpts = sorted(glob.glob(f'{drive_backup}/checkpoint_*.pt'))
if not local_ckpts and drive_ckpts:
    os.makedirs('/content/outputs/checkpoints', exist_ok=True)
    shutil.copy2(drive_ckpts[-1], '/content/outputs/checkpoints/')
    print(f"Restored: {os.path.basename(drive_ckpts[-1])}")
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 5 --resume {resume}

In [ ]:
listen_podcast(5)  # Expect: clear Indian English!
print("\n--- FINAL COMPARISON: ---")
compare_podcasts()

---
## Generate Custom Speech

Use the model on any text you want.

In [ ]:
from indian_tts.inference import IndianTTS
import IPython.display as ipd

ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
tts = IndianTTS(ckpts[-1])

# Try your own text!
my_text = "Hello, this is my custom Indian text to speech model speaking."

for voice in ['male', 'female']:
    audio = tts.synthesize(my_text, voice=voice)
    print(f"\n[{voice.upper()}] {my_text}")
    ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

In [ ]:
# Generate a fresh podcast with different settings
!python -m indian_tts.podcast_demo \
    --checkpoint {ckpts[-1]} \
    --output /content/outputs/podcast_custom \
    --speed 1.0 \
    --expressiveness 0.8

ipd.display(ipd.Audio('/content/outputs/podcast_custom/podcast_full.wav'))